# Historical Rockstar HLIST selection — Milky Way-est

This notebook reproduces the **downstream historical association reconstruction** used in the corrected MWest analysis. It is deliberately separate from the `symlib`/Symfind selection notebook.

**Input:** corrected raw-HLIST node tables in `lmc_associated_catalog_raw/`, generated upstream. This notebook **does not** recreate those node tables from native Rockstar HLIST files. It reconstructs historical association and peak mass from the available nodes and exports pre-infall selections. The original workflow appears in `LMC_MAH_OJA_single_column(1).ipynb`, especially the corrected radial-CDF analysis.

A halo is associated if it lies inside the LMC virial radius at a pre-infall snapshot **or** inherits that status from an associated progenitor through `desc_id`. No post-infall capture is allowed into the pre-infall cohort. All merger-tree IDs are read as strings to avoid precision loss.

## 1. Paths and inputs

Set `RAW_DIR` to your existing corrected catalog directory. You only need `numpy`, `pandas`, and optionally `symlib` plus `lmc_reference.py` when the tables do not contain `pre_or_at_infall`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RAW_DIR = Path("lmc_associated_catalog_raw")
OUT_DIR = Path("historical_hlist_selection")
OUT_DIR.mkdir(parents=True, exist_ok=True)
BASE_MWEST = "/data/darkmatter/programs/symlib/standard/"

REQUIRED = {"raw_snapshot", "a", "halo_id", "desc_id", "Mvir",
            "inside_Rvir_LMC", "r_over_Rvir_LMC"}
CANDIDATE_SUFFIXES = (
    "_associated_nodes_with_mpeak.csv.gz",
    "_associated_nodes.csv.gz",
    "_associated_nodes_phase_space.csv.gz",
)

def read_host(host):
    for suffix in CANDIDATE_SUFFIXES:
        path = RAW_DIR / f"{host}{suffix}"
        if not path.is_file():
            continue
        cols = set(pd.read_csv(path, nrows=0).columns)
        if REQUIRED <= cols:
            return pd.read_csv(path, dtype={"halo_id": "string", "desc_id": "string"}), path
    raise FileNotFoundError(f"{host}: missing corrected node table with {sorted(REQUIRED)}")

print(f"Raw directory: {RAW_DIR.resolve()}")

## 2. Catalog conventions

The filters below mirror the corrected analysis. The `physical` / `is_physical` flags take priority; otherwise `phantom == 0` is used. If none of those columns exist, no physical-object cut can be inferred. An absent flag is not evidence that every object is physically reliable.

In [ ]:
def as_bool(values):
    s = pd.Series(values)
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).to_numpy(bool)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).to_numpy(float) != 0
    return s.astype(str).str.strip().str.lower().isin(["true", "t", "1", "yes", "y"]).to_numpy(bool)


def physical_mask(d):
    if "physical" in d.columns:
        return as_bool(d["physical"])
    if "is_physical" in d.columns:
        return as_bool(d["is_physical"])
    if "phantom" in d.columns:
        return pd.to_numeric(d["phantom"], errors="coerce").fillna(1).to_numpy(float) == 0
    return np.ones(len(d), dtype=bool)


def id_key(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    try:
        return int(text)
    except ValueError:
        return None


def preinfall_epoch(data):
    if "pre_or_at_infall" not in data.columns:
        return None
    pre = data.loc[as_bool(data["pre_or_at_infall"]), ["raw_snapshot", "a"]]
    pre = pre.dropna().drop_duplicates("raw_snapshot")
    if pre.empty:
        return None
    row = pre.sort_values("a").iloc[-1]
    return int(row["raw_snapshot"]), float(row["a"])

## 3. Identify the last pre-infall snapshot

Prefer the `pre_or_at_infall` flag in the corrected raw table. If it is missing, use the original LMC index mapping and the first MW-host virial-radius crossing from `symlib` to identify the target scale factor; map this to the nearest raw HLIST snapshot. This fallback is only used when necessary, and the selected epoch should be checked against the raw HLIST sampling.

In [ ]:
def epoch_for_host(host, data):
    epoch = preinfall_epoch(data)
    if epoch is not None:
        return (*epoch, "raw flag")

    import symlib
    from lmc_reference import lmc_index_by_host, find_pre_infall_snapshot

    lookup = {Path(symlib.get_host_directory(BASE_MWEST, "MWest", i)).name: i
              for i in range(symlib.n_hosts("MWest"))}
    i_host = lookup[host]
    h, _ = symlib.read_subhalos(symlib.get_host_directory(BASE_MWEST, "MWest", i_host))
    scale = np.asarray(symlib.scale_factors(
        symlib.get_host_directory(BASE_MWEST, "MWest", i_host)), float)
    i_pre = find_pre_infall_snapshot(h[0], h[lmc_index_by_host[i_host]], mode="crossing")
    if i_pre is None:
        raise RuntimeError(f"{host}: LMC infall crossing not found")
    epochs = data[["raw_snapshot", "a"]].dropna().drop_duplicates("raw_snapshot")
    row = epochs.iloc[np.argmin(abs(epochs["a"].to_numpy(float) - scale[i_pre]))]
    return int(row["raw_snapshot"]), float(row["a"]), "symlib crossing fallback"

## 4. Reconstruct association and historical peak mass

Process snapshots in chronological order, stopping at the last pre-infall snapshot. At each node, association is `inside_Rvir_LMC` **or** inherited from an associated progenitor. Historical peak mass is the maximum `Mvir` along the available progenitor links, never a mass obtained from future snapshots. Multiple progenitors mapping to one descendant are merged using logical OR for association and maximum for peak mass.

As in the original corrected analysis, inheritance here moves across successive *available* raw snapshots. Missing progenitor rows can therefore make membership or peak masses incomplete.

In [ ]:
def causal_preinfall_selection(d, pre_snap):
    data = d.loc[d["raw_snapshot"].notna() & d["a"].notna()].copy()
    data = data.sort_values(["a", "raw_snapshot"])
    data = data.drop_duplicates(["raw_snapshot", "halo_id"], keep="last")
    data["inside_flag"] = as_bool(data["inside_Rvir_LMC"])
    data["rnorm"] = pd.to_numeric(data["r_over_Rvir_LMC"], errors="coerce")
    data["Mvir_now"] = pd.to_numeric(data["Mvir"], errors="coerce")
    data["physical_flag"] = physical_mask(data)

    snaps = data[["raw_snapshot", "a"]].drop_duplicates("raw_snapshot").sort_values("a")
    inherited_assoc, inherited_peak = set(), {}
    for item in snaps.itertuples(index=False):
        snap = int(item.raw_snapshot)
        block = data.loc[data["raw_snapshot"] == snap].copy()
        assoc = np.zeros(len(block), dtype=bool)
        mpeak = np.full(len(block), np.nan)
        next_assoc, next_peak = set(), {}
        for j, (hid, did, mass, inside) in enumerate(zip(
                block["halo_id"], block["desc_id"], block["Mvir_now"], block["inside_flag"])):
            key = id_key(hid)
            if key is None:
                continue
            assoc[j] = bool(inside) or key in inherited_assoc
            prior = inherited_peak.get(key, np.nan)
            valid = [v for v in (mass, prior) if np.isfinite(v) and v > 0]
            if valid:
                mpeak[j] = max(valid)
            dk = id_key(did)
            if dk is not None and dk >= 0:
                if assoc[j]:
                    next_assoc.add(dk)
                if np.isfinite(mpeak[j]):
                    next_peak[dk] = max(next_peak.get(dk, 0.0), mpeak[j])
        if snap == pre_snap:
            block["causal_assoc"] = assoc
            block["Mpeak_upto_infall"] = mpeak
            return block
        inherited_assoc, inherited_peak = next_assoc, next_peak
    raise RuntimeError(f"Infall snapshot {pre_snap} not present")

## 5. Inspect one host

The next cell produces the uncut pre-infall association table. The LMC itself can be included in the raw nodes and is excluded from the *subhalo sample* below using the same positional/relative-mass proxy as the corrected radial-CDF analysis.

In [ ]:
HOST = "Halo170"  # Change to any MWest host
raw, path = read_host(HOST)
pre_snap, a_pre, epoch_source = epoch_for_host(HOST, raw)
pre = causal_preinfall_selection(raw, pre_snap)
print(f"{HOST}: {path.name} | raw snap={pre_snap}, a={a_pre:.5f} ({epoch_source})")
print(f"Rows at infall={len(pre)}, causally associated={int(pre.causal_assoc.sum())}")
display(pre[["halo_id", "desc_id", "causal_assoc", "inside_flag",
             "Mvir_now", "Mpeak_upto_infall", "rnorm"]].head(15))

## 6. Optional analysis cuts (not part of association itself)

The SHMF/radial-CDF analysis additionally required physical objects, excluded the central LMC itself, imposed a mass-resolution limit, and restricted the instantaneous infall position to `r/Rvir,LMC < 1`. **Do not treat these cuts as the definition of association.**

For a standalone handoff, `selected_at_infall` below defaults to the **causal association list**; `sample_for_radial_cdf` applies the original radial-CDF-specific cuts when a mass threshold is supplied. The analysis used the maximum of each host's native 300-particle threshold and the median native threshold over MWest hosts. Those thresholds come from the corrected `all_hosts_catalog_summary*.csv` files.

In [ ]:
def read_resolution_limits():
    for name in ("all_hosts_catalog_summary_with_mpeak.csv", "all_hosts_catalog_summary.csv"):
        p = RAW_DIR / name
        if p.exists():
            table = pd.read_csv(p)
            if {"host_name", "Mres_300p"} <= set(table.columns):
                values = table.set_index("host_name")["Mres_300p"]
                return {str(k): float(v) for k, v in values.items() if np.isfinite(v) and v > 0}
    return {}


def selected_at_infall(pre):
    return pre.loc[pre["causal_assoc"]].copy()


def sample_for_radial_cdf(pre, mass_cut):
    r = pre["rnorm"].to_numpy(float)
    peak = pre["Mpeak_upto_infall"].to_numpy(float)
    assoc = pre["causal_assoc"].to_numpy(bool)
    mu = pd.to_numeric(pre["mu_Mvir"], errors="coerce").to_numpy(float) \
         if "mu_Mvir" in pre else np.full(len(pre), np.nan)
    lmc_self = np.isfinite(r) & (abs(r) < 1e-8) & np.isfinite(mu) & (mu > 0.5)
    keep = (assoc & pre["physical_flag"].to_numpy(bool) & ~lmc_self &
            np.isfinite(peak) & (peak >= mass_cut) & np.isfinite(r) & (r >= 0) & (r < 1))
    return pre.loc[keep].copy()

limits = read_resolution_limits()
print("Hosts with native 300-particle limits:", len(limits))
if HOST in limits:
    mass_cut = max(limits[HOST], np.median(list(limits.values())))
    print("Radial-CDF sample size:", len(sample_for_radial_cdf(pre, mass_cut)))
else:
    print("No verified 300-particle threshold: mass-cut sample not computed")

## 7. Export selection tables for all hosts

These CSVs identify **raw historical HLIST nodes at the pre-infall epoch**, not symlib branch indices. Keep `halo_id` as an exact identifier; do not join it directly to a Symfind or Rockstar `sub_idx` without a verified cross-match. The full uncut association list is exported separately from any radial-CDF-filtered list.

In [ ]:
HOSTS = sorted({p.name.split("_associated_nodes")[0]
                for p in RAW_DIR.glob("Halo*_associated_nodes*.csv.gz")})
if not HOSTS:
    raise FileNotFoundError(f"No corrected host tables in {RAW_DIR.resolve()}")

all_members, summaries, errors = [], [], []
limits = read_resolution_limits()
median_cut = np.median(list(limits.values())) if limits else np.nan
for host in HOSTS:
    try:
        raw, path = read_host(host)
        snap, a_pre, how = epoch_for_host(host, raw)
        infall = causal_preinfall_selection(raw, snap)
        associated = selected_at_infall(infall).copy()
        associated.insert(0, "host_name", host)
        associated["a_pre"] = a_pre
        associated["epoch_source"] = how
        all_members.append(associated)
        summary = {"host_name": host, "raw_snapshot": snap, "a_pre": a_pre,
                   "epoch_source": how, "n_nodes": len(infall),
                   "n_causal_associated": len(associated), "catalog": path.name}
        if host in limits:
            cut = max(limits[host], median_cut)
            summary["effective_radial_cdf_mass_cut"] = cut
            summary["n_radial_cdf_sample"] = len(sample_for_radial_cdf(infall, cut))
        summaries.append(summary)
    except (FileNotFoundError, KeyError, ValueError, RuntimeError) as exc:
        errors.append({"host_name": host, "error": str(exc)})

if all_members:
    # Preserve ID strings when opening the exported CSV with pandas.
    pd.concat(all_members, ignore_index=True).to_csv(
        OUT_DIR / "mwest_raw_hlist_causal_members_at_infall.csv.gz", index=False, compression="gzip")
pd.DataFrame(summaries).to_csv(OUT_DIR / "mwest_raw_hlist_host_summary.csv", index=False)
pd.DataFrame(errors, columns=["host_name", "error"]).to_csv(
    OUT_DIR / "mwest_raw_hlist_skipped_hosts.csv", index=False)
print(f"Exported {len(summaries)} hosts; skipped {len(errors)}")
display(pd.DataFrame(summaries).head())
if errors:
    display(pd.DataFrame(errors))

## Scope and caveats

- **Catalog construction is upstream.** The original script converting native Rockstar HLIST snapshots into the corrected `lmc_associated_catalog_raw/` tables has not been supplied here. This notebook therefore does not establish how those raw nodes were initially gathered, which halo centers and unit conversions were used, or whether the supplied catalogs contain every possible progenitor.
- **Pre-infall selection versus later propagation.** This notebook exports the causal membership at infall. The corrected plots tracing the fixed infall cohort to later times additionally follow descendant IDs after infall (with no post-infall recruitment); consult the full original analysis for those endpoint calculations.
- **Not the Symfind sample.** Symfind follows tracked cores and has its own availability/completeness. Do not compare the two catalogs row-by-row without matching objects.
- **Catalog IDs versus symlib indices.** `halo_id` and `desc_id` are Rockstar historical IDs, not the `sub_idx` values in the other handoff.
- **Data dependence.** Missing raw nodes cannot be recovered by this notebook. Invalid IDs and missing epochs are reported, not silently repaired.

**Analysis provenance:** `LMC_MAH_OJA_single_column(1).ipynb`, corrected raw-HLIST radial-CDF and infall-to-present analysis cells. Use alongside `MWest_LMC_selection.ipynb` for the separate symlib/Symfind method.